# Memory Systems in CrewAI

> **MLCourse - Agentic AI - CrewAI Advanced Agents - Module 03**

## What you will learn

- `MemoryConfig`: the configuration object that defines what memory types are active.
- Short-term memory: conversation history within a single crew execution.
- Long-term memory: persistent storage across multiple crew executions.
- Entity memory: tracks people, organizations, and concepts mentioned in conversations.
- Contextual memory: combines short-term and entity memory for rich context.
- How memory persists: run a crew twice and show context carrying over.

Memory is what turns a stateless LLM call into a stateful assistant. Without memory,
every crew execution starts from zero. With memory, the agent remembers what it
learned in previous runs -- just like a human colleague who recalls yesterday's meeting.

In [ ]:
# --- Standard library imports -------------------------------------------------
import os                # Environment variable access.
from pathlib import Path # Path objects for filesystem navigation.
import json             # For pretty-printing memory contents.

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv  # Loads KEY=value lines from .env into os.environ.

# Walk up from wherever this notebook was launched until we reach the track root
# folder "03_agentic_ai". This makes the notebook runnable from any subfolder.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

# Jupyter plotting magic, wrapped so this file stays valid pure Python when it is
# executed as a plain script or checked by automated QA tooling outside IPython.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass  # Not running inside IPython -- nothing to configure.

print("Setup complete. Track root resolved to:", TRACK)

In [ ]:
# --- CrewAI imports -----------------------------------------------------------
from crewai import Agent, Task, Crew, Process
from crewai import MemoryConfig  # Configuration for memory systems.
from langchain_ollama import ChatOllama  # Local LLM -- no API key needed.

print("CrewAI memory classes imported.")

## 1. Memory types overview

CrewAI provides three distinct memory types, each serving a different purpose:

| Memory Type | Scope | Persistence | Use Case |
|---|---|---|---|
| **Short-term** | Single execution | In-memory, lost after crew finishes | Conversation flow within one run |
| **Long-term** | Cross-execution | Stored on disk (SQLite) | Learning from past interactions |
| **Entity** | Cross-execution | Stored on disk (SQLite) | Tracking people, orgs, concepts |

All three are configured via `MemoryConfig` and enabled at the Crew level. The
crew manages memory automatically -- agents read from and write to memory without
explicit API calls.

> **Key insight:** short-term memory is like RAM (fast, temporary), long-term
> memory is like a hard drive (persistent, slower), and entity memory is like a
> knowledge graph (structured relationships between things).

In [ ]:
# MemoryConfig defines which memory types are active and where they store data.
# The simplest config enables all three types with default storage paths.
memory_config = MemoryConfig(
    # Enable short-term memory -- stores conversation history within one execution.
    short_term=True,
    # Enable long-term memory -- stores learnings across executions.
    long_term=True,
    # Enable entity memory -- tracks named entities (people, orgs, concepts).
    entity=True,
)

print("MemoryConfig created with all three memory types enabled.")
print(f"  Short-term: {memory_config.short_term}")
print(f"  Long-term:  {memory_config.long_term}")
print(f"  Entity:     {memory_config.entity}")

## 2. Short-term memory -- within a single execution

Short-term memory is the simplest type. It stores the conversation history
(all messages exchanged between agents and the environment) during a single
`crew.kickoff()` call. When the crew finishes, short-term memory is discarded.

Think of it as the agent's working memory -- what it has said and heard in the
current session. It enables multi-turn reasoning within a single execution:
Agent A says something, Agent B responds to it, and so on.

Short-term memory is enabled by default in CrewAI. You do not need to configure
anything to use it -- it just works.

In [ ]:
# Simple crew without explicit memory config -- short-term is on by default.
llm = ChatOllama(model="llama3.1:8b", temperature=0)

memoryless_agent = Agent(
    role="Conversational Assistant",
    goal="Answer questions concisely, remembering what was said earlier in the conversation.",
    backstory="You are a helpful assistant that builds on previous context.",
    llm=llm,
    verbose=True,
    # No memory= parameter means default (short-term only).
)

In [ ]:
# Two tasks that build on each other -- short-term memory lets the agent
# remember the context from task 1 when executing task 2.
context_task_1 = Task(
    description=(
        "Introduce yourself and explain what CrewAI is in exactly two sentences. "
        "Remember this explanation -- the next task will reference it."
    ),
    expected_output="A two-sentence explanation of CrewAI.",
    agent=memoryless_agent,
)

context_task_2 = Task(
    description=(
        "Based on your earlier explanation, now expand on ONE specific feature "
        "of CrewAI that you mentioned. Go into detail about how it works."
    ),
    expected_output="A detailed expansion on one CrewAI feature from your earlier explanation.",
    agent=memoryless_agent,
)

In [ ]:
short_term_crew = Crew(
    agents=[memoryless_agent],
    tasks=[context_task_1, context_task_2],
    process=Process.sequential,
    verbose=True,
    memory=True,  # Explicitly enable memory (short-term by default).
)

print("=== Short-term Memory Demo ===")
print("Task 2 should reference what was said in Task 1.\n")
try:
    result = short_term_crew.kickoff()
    print("\n=== Result ===")
    print(result)
except Exception as e:
    print(f"[crew error] {e}")
    print("Ensure Ollama is running: ollama serve && ollama pull llama3.1:8b")

## 3. Long-term memory -- persistence across executions

Long-term memory stores agent learnings in a SQLite database on disk. Unlike
short-term memory, it PERSISTS between crew executions. When you run the crew
again, the agent can recall what it learned in previous runs.

This is powerful for:
- Iterative refinement: agent improves its answers over multiple runs.
- User profiling: agent remembers user preferences across sessions.
- Knowledge accumulation: agent builds up expertise over time.

Long-term memory uses LLM-generated summaries of past interactions, stored as
text in SQLite. At retrieval time, relevant summaries are injected into the
agent's context alongside the current conversation.

In [ ]:
# Configure memory with long-term persistence enabled.
# The storage_path determines where the SQLite database is created.
LONG_TERM_DB = Path.cwd() / "crewai_memory_demo"
LONG_TERM_DB.mkdir(exist_ok=True)

long_term_config = MemoryConfig(
    short_term=True,
    long_term=True,       # Enable cross-execution persistence.
    entity=True,
)

In [ ]:
# Create an agent that will be used across two separate executions.
planning_agent = Agent(
    role="Project Planner",
    goal="Plan projects by considering past planning decisions and lessons learned.",
    backstory=(
        "You are an experienced project planner. You remember what worked and "
        "what did not in previous planning sessions. You build on past experience."
    ),
    llm=ChatOllama(model="llama3.1:8b", temperature=0),
    verbose=True,
)

In [ ]:
# Task for the first execution -- initial project plan.
plan_task = Task(
    description=(
        "Create a project plan for building a customer support chatbot. "
        "Include: 1) key milestones, 2) estimated timeline, 3) major risks. "
        "Keep it concise -- 3 bullet points for each section."
    ),
    expected_output="A concise project plan with milestones, timeline, and risks.",
    agent=planning_agent,
)

In [ ]:
# First crew execution -- this will store learnings in long-term memory.
lt_crew_run1 = Crew(
    agents=[planning_agent],
    tasks=[plan_task],
    process=Process.sequential,
    memory=True,
    verbose=True,
)

print("=== Long-term Memory: Run 1 (Initial Plan) ===")
try:
    result_1 = lt_crew_run1.kickoff()
    print("\n=== Plan from Run 1 ===")
    print(result_1)
except Exception as e:
    print(f"[crew error] {e}")
    print("Ensure Ollama is running: ollama serve && ollama pull llama3.1:8b")

## 4. Second execution -- memory carries context

Now run a DIFFERENT crew (new instance) with the SAME agent but long-term memory
enabled. The agent should recall insights from the first run and incorporate them
into the revised plan.

This demonstrates the key value of long-term memory: the agent does not start
from zero. It builds on accumulated experience, just like a human planner who
remembers their previous proposals.

In [ ]:
# Task for the second execution -- revised plan.
revised_task = Task(
    description=(
        "Revise your earlier project plan for the customer support chatbot. "
        "Consider what you previously proposed and improve it based on any "
        "lessons learned or insights from your long-term memory. "
        "Highlight what changed and why."
    ),
    expected_output="A revised project plan with explicit changes from the previous version.",
    agent=planning_agent,
)

In [ ]:
# Second crew execution -- should reference Run 1 via long-term memory.
lt_crew_run2 = Crew(
    agents=[planning_agent],
    tasks=[revised_task],
    process=Process.sequential,
    memory=True,
    verbose=True,
)

print("=== Long-term Memory: Run 2 (Revised Plan) ===")
print("Agent should recall and build on Run 1's plan.\n")
try:
    result_2 = lt_crew_run2.kickoff()
    print("\n=== Revised Plan from Run 2 ===")
    print(result_2)
except Exception as e:
    print(f"[crew error] {e}")
    print("Ensure Ollama is running: ollama serve && ollama pull llama3.1:8b")

## 5. Entity memory -- tracking named things

Entity memory tracks named entities (people, organizations, technologies,
concepts) mentioned across conversations. It builds a structured view of
"who knows what" and "what relates to what."

When an agent mentions "Alice from the data team" in one task, entity memory
records:
- Entity: Alice
- Role: data team member
- Context: mentioned in project planning discussion

In a later task, if the agent needs to assign data work, entity memory helps
it recall that Alice exists and what her role is.

Entity memory is particularly useful in multi-agent crews where agents need
to coordinate around shared entities.

In [ ]:
# Agent with entity memory enabled.
entity_agent = Agent(
    role="Team Coordinator",
    goal="Coordinate team members and track who is working on what.",
    backstory=(
        "You keep track of team members, their skills, and their current assignments. "
        "You remember people mentioned in previous discussions."
    ),
    llm=ChatOllama(model="llama3.1:8b", temperature=0),
    verbose=True,
)

In [ ]:
# Task 1: introduce team members.
intro_task = Task(
    description=(
        "Our project team includes: Alice (frontend developer, 5 years React), "
        "Bob (backend engineer, expert in Python and FastAPI), "
        "Carol (ML engineer, specializes in NLP and transformer models). "
        "Introduce the team and their roles."
    ),
    expected_output="A summary of each team member and their role.",
    agent=entity_agent,
)

In [ ]:
# Task 2: assign work based on team composition.
# Entity memory should help the agent recall who is available and their skills.
assignment_task = Task(
    description=(
        "We need to build an NLP-powered search feature. Based on the team "
        "members you know about, suggest task assignments and explain your reasoning."
    ),
    expected_output="Task assignments with reasoning based on team member skills.",
    agent=entity_agent,
)

In [ ]:
entity_crew = Crew(
    agents=[entity_agent],
    tasks=[intro_task, assignment_task],
    process=Process.sequential,
    memory=True,  # Enables all memory types including entity memory.
    verbose=True,
)

print("=== Entity Memory Demo ===")
print("Task 2 should reference team members from Task 1.\n")
try:
    result = entity_crew.kickoff()
    print("\n=== Result ===")
    print(result)
except Exception as e:
    print(f"[crew error] {e}")
    print("Ensure Ollama is running: ollama serve && ollama pull llama3.1:8b")

## 6. Contextual memory -- the combined view

Contextual memory is not a separate storage type -- it is the COMBINATION of
short-term and entity memory that the agent sees at query time. When an agent
processes a task, CrewAI merges:

1. **Short-term memory**: recent conversation turns (what was just discussed).
2. **Entity memory**: structured knowledge about named entities.
3. **Long-term memory**: summaries of past executions (if enabled).

The merged context is prepended to the agent's prompt. This gives the agent a
rich, multi-layered view of the conversation -- recent dialogue, named entities,
and historical knowledge all in one context window.

> **Pro tip:** contextual memory quality depends on memory size. If the context
> window fills up, older memories get truncated. Keep entity descriptions concise
> and long-term summaries focused.

In [ ]:
# Demonstrate contextual memory with a multi-agent crew.
researcher = Agent(
    role="Research Analyst",
    goal="Research topics thoroughly and provide detailed analysis.",
    backstory="You are a meticulous researcher who builds on prior findings.",
    llm=ChatOllama(model="llama3.1:8b", temperature=0),
)

writer = Agent(
    role="Technical Writer",
    goal="Write clear, concise technical documentation.",
    backstory="You turn research findings into accessible documentation.",
    llm=ChatOllama(model="llama3.1:8b", temperature=0),
)

In [ ]:
research_task = Task(
    description=(
        "Research the key benefits of multi-agent systems: 1) specialization, "
        "2) parallel execution, 3) fault tolerance. Provide 2 sentences per benefit."
    ),
    expected_output="A concise research summary covering three benefits of multi-agent systems.",
    agent=researcher,
)

writing_task = Task(
    description=(
        "Based on the research findings, write a brief executive summary "
        "(3-4 sentences) explaining why multi-agent systems are valuable."
    ),
    expected_output="An executive summary based on the research findings.",
    agent=writer,
)

In [ ]:
contextual_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    memory=True,
    verbose=True,
)

print("=== Contextual Memory Demo ===")
print("Writer should reference researcher's findings (short-term memory).\n")
try:
    result = contextual_crew.kickoff()
    print("\n=== Result ===")
    print(result)
except Exception as e:
    print(f"[crew error] {e}")
    print("Ensure Ollama is running: ollama serve && ollama pull llama3.1:8b")

## 7. Memory persistence across separate executions

The real power of long-term memory shows when you create an entirely NEW crew
instance (new Python objects) but the memory persists via the underlying SQLite
database. Below we simulate this by running two separate crews and comparing outputs.

In production, this means:
- A customer support agent remembers a user's previous tickets.
- A coding assistant remembers the project's conventions.
- A research assistant remembers papers it has already reviewed.

In [ ]:
print("=== Memory Persistence Across Executions ===")
print()
print("Demonstration flow:")
print("  1. Run 1: Agent learns about a project (memory stored to disk).")
print("  2. Run 2: NEW crew instance, SAME memory path.")
print("  3. Agent should reference Run 1 context in Run 2.")
print()

# Run 1 -- agent learns about Project Alpha.
learn_task = Task(
    description=(
        "Project Alpha is a machine learning pipeline that processes satellite imagery "
        "to detect deforestation. The stack uses PyTorch, rasterio for geospatial data, "
        "and PostgreSQL with PostGIS for storage. The team has 3 ML engineers and "
        "2 backend developers. The project deadline is Q4 2026."
    ),
    expected_output="A confirmation that you have noted the project details.",
    agent=planning_agent,
)

run1_crew = Crew(
    agents=[planning_agent],
    tasks=[learn_task],
    process=Process.sequential,
    memory=True,
    verbose=True,
)

try:
    result_1 = run1_crew.kickoff()
    print("Run 1 complete -- project details stored in long-term memory.\n")
except Exception as e:
    print(f"[crew error] {e}")

## 8. Run 2 -- recall from a fresh crew

Create an entirely new crew with new Agent/Task objects but the same underlying
memory storage. The agent should be able to recall Project Alpha details.

In [ ]:
# New planning task that references the previously learned project.
recall_task = Task(
    description=(
        "What project do you know about from your memory? Describe its technology "
        "stack, team composition, and deadline. Also suggest next steps based on "
        "what you know."
    ),
    expected_output="A description of the project from memory, with suggested next steps.",
    agent=planning_agent,
)

run2_crew = Crew(
    agents=[planning_agent],
    tasks=[recall_task],
    process=Process.sequential,
    memory=True,
    verbose=True,
)

print("=== Run 2: Recall from Memory ===")
print("Agent should recall Project Alpha details from Run 1.\n")
try:
    result_2 = run2_crew.kickoff()
    print("\n=== Recall Result ===")
    print(result_2)
except Exception as e:
    print(f"[crew error] {e}")
    print("Ensure Ollama is running: ollama serve && ollama pull llama3.1:8b")

## Cleanup

Remove the demo memory database. In production, you would keep the SQLite file
for persistent memory across sessions.

In [ ]:
# Clean up the memory database directory.
import shutil
if LONG_TERM_DB.exists():
    shutil.rmtree(LONG_TERM_DB)
    print(f"Cleaned up: {LONG_TERM_DB}")

## Summary and key takeaways

- `MemoryConfig`: configure which memory types are active (short_term, long_term, entity).
- **Short-term memory**: conversation history within one execution, automatic, ephemeral.
- **Long-term memory**: SQLite-backed persistence across executions, stores LLM summaries.
- **Entity memory**: tracks named entities (people, orgs, concepts) across conversations.
- **Contextual memory**: merged view of all memory types, injected into agent prompts.
- Memory persists across NEW crew instances if they share the same storage path.
- `memory=True` on Crew enables all memory types; `MemoryConfig` for fine-grained control.

**Next up:** Module 04 covers Reasoning and Planning -- chain-of-thought and task decomposition.